# 02_wall_cft_windows: independent resumable A100 bundle

raw hard/soft programmable-wall window spectra, contours, and squared correlators. This notebook launches only `02_wall_cft_windows`; it never continues into another experiment. The sampling layout uses 25 independent trajectories in five immutable five-trajectory shards. Run the default report-only pass first, inspect the exact verified/pending inventory, then set `RESUME_REPORT_ONLY=False` to compute only missing shards.

## 1. Mount Drive and locate the campaign package


In [ ]:
from pathlib import Path
import hashlib, importlib.util, json, subprocess, sys

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

MYDRIVE = Path('/content/drive/MyDrive')
CAMPAIGN_ROOT = MYDRIVE / 'final_production_new_designs'
runner = CAMPAIGN_ROOT / 'colab_bundle_runner.py'
bundle_root = CAMPAIGN_ROOT / '02_wall_cft_windows'
if not runner.is_file() or not (bundle_root / 'run_bundle.py').is_file():
    raise FileNotFoundError('Upload this bundle with the final_production_new_designs parent support files')
def _load_runner_module(module_path, module_name):
    module_spec = importlib.util.spec_from_file_location(module_name, module_path)
    if module_spec is None or module_spec.loader is None:
        raise ImportError(f'Cannot load runner: {module_path}')
    module = importlib.util.module_from_spec(module_spec)
    sys.modules[module_spec.name] = module
    module_spec.loader.exec_module(module)
    return module
runner_module = _load_runner_module(runner, '_colab_bundle_runner')
bundle_config = json.loads((bundle_root / 'production_config.json').read_text())
OUTPUT_BUNDLE = bundle_config.get('output_bundle', bundle_root.name)
print(f'[campaign] {CAMPAIGN_ROOT}')
print(f'[bundle] {bundle_root}')


## 2. Select this bundle's queue


In [ ]:
RUN_PROFILE = 'production'
BUNDLE = '02_wall_cft_windows'
CASE_PREFIXES = ()  # e.g. ('S2_',); empty runs every currently eligible case
CASE_IDS = ()       # exact IDs; empty uses prefixes/full bundle
MAX_SESSION_GPU_HOURS_OVERRIDE = None
PREFLIGHT_ONLY = False
RESUME_REPORT_ONLY = True  # inspect first; change to False to launch missing work
HEARTBEAT_SECONDS = 60
RUN_QUEUE = True
if HEARTBEAT_SECONDS <= 0:
    raise ValueError('HEARTBEAT_SECONDS must be positive')
if CASE_PREFIXES and CASE_IDS:
    raise ValueError('use CASE_PREFIXES or CASE_IDS, not both')
print(json.dumps({
    'bundle': BUNDLE, 'profile': RUN_PROFILE, 'case_prefixes': CASE_PREFIXES,
    'case_ids': CASE_IDS, 'resume_report_only': RESUME_REPORT_ONLY,
    'heartbeat_seconds': HEARTBEAT_SECONDS,
}, indent=2))


## 3. Required Ny=60 hard/soft A100 preflight

The production queue now runs or reuses this qualification automatically before its first missing shard. This optional cell lets you run it early and inspect the receipt. It evaluates a complete five-trajectory shard for each wall construction at the largest geometry, including all Ay=30 origins, and writes a version-matched safety receipt. No scientific parameter is reduced if the preflight is unsafe.


In [ ]:
RUN_A100_PREFLIGHT = False
if RUN_A100_PREFLIGHT:
    preflight_module = _load_runner_module(bundle_root / 'run_bundle.py', '_colab_wall_preflight_runner')
    preflight_argv = ['--drive-root', str(MYDRIVE), '--mode', 'production', '--a100-preflight']
    returncode = preflight_module.main(preflight_argv)
    if returncode not in (None, 0):
        raise RuntimeError(f'Preflight runner returned nonzero status {returncode}')


## 3. Verify, resume, and stop at the end of this bundle


In [ ]:
if not RUN_QUEUE:
    raise RuntimeError('RUN_QUEUE is False')
runner_argv = [
    '--drive-root', str(MYDRIVE),
    '--bundle', BUNDLE, '--profile', RUN_PROFILE,
    '--heartbeat-seconds', str(HEARTBEAT_SECONDS),
]
for prefix in CASE_PREFIXES:
    runner_argv += ['--case-prefix', str(prefix)]
for case_id in CASE_IDS:
    runner_argv += ['--case-id', str(case_id)]
if MAX_SESSION_GPU_HOURS_OVERRIDE is not None:
    runner_argv += ['--max-session-hours', str(MAX_SESSION_GPU_HOURS_OVERRIDE)]
if PREFLIGHT_ONLY:
    runner_argv.append('--preflight-only')
if RESUME_REPORT_ONLY:
    runner_argv.append('--resume-report-only')
command = [sys.executable, '-u', str(runner), *runner_argv]
production_collection = bundle_config.get('production_output_collection', 'classA_final_production_outputs/production_10sample_v4_occupied_frame_cycle_resolved')
pilot_collection = bundle_config.get('pilot_output_collection', 'classA_pilot_outputs/production_10sample_v4_occupied_frame_cycle_resolved')
collection = production_collection if RUN_PROFILE == 'production' else pilot_collection
output_root = MYDRIVE / collection
session_dir = output_root / '_bundle_sessions'
build_id = hashlib.sha256(runner.read_bytes()).hexdigest()[:16]
print('[bundle dashboard]')
print(json.dumps({
    'bundle': BUNDLE, 'profile': RUN_PROFILE, 'output_root': str(output_root),
    'runner_build_id': build_id, 'heartbeat_seconds': HEARTBEAT_SECONDS,
    'session_dir': str(session_dir), 'case_prefixes': CASE_PREFIXES,
    'case_ids': CASE_IDS, 'resume_report_only': RESUME_REPORT_ONLY,
}, indent=2))
print(f'[launch bundle] {" ".join(command)}', flush=True)
try:
    returncode = runner_module.main(runner_argv)
    if returncode not in (None, 0):
        raise RuntimeError(f'Runner returned nonzero status {returncode}')
except BaseException:
    sessions = sorted(session_dir.glob(f'*_{BUNDLE}_*.json'), key=lambda p: p.stat().st_mtime, reverse=True)
    if sessions:
        latest = json.loads(sessions[0].read_text())
        print('[saved bundle failure]')
        print(json.dumps({'session': str(sessions[0]), 'failure': latest.get('failure'), 'current': latest.get('current'), 'log': latest.get('session_log')}, indent=2))
    raise


## Final. Disconnect this completed Colab runtime


In [ ]:
from google.colab import runtime
runtime.unassign()
print('done')
